# SatQuery AI — GPU Training Notebook

Trains all six models used by the SatQuery AI backend, on a free Colab GPU:

1. **Ship classifier** — ShipsNet (CNN, ship/no-ship)
2. **Aerial multi-class detector** — NWPU VHR-10 (Faster R-CNN, 10 classes)
3. **Caption fine-tune** — RSICD (BLIP captioning)
4. **VQA fine-tune** — RSVQA-HR (BLIP-VQA)
5. **Phase-2 detector** — DOTA v1.5 (Faster R-CNN, 16 classes, oriented boxes simplified to axis-aligned)
6. **Learned change detection** — LEVIR-CD (U-Net, replaces classical image diffing)

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (free tier), then `Runtime -> Run all`.

Everything checkpoints to Google Drive every ~50 batches, so if the Colab runtime
disconnects or hits its time limit, just reconnect and re-run this notebook —
each section reloads its last saved checkpoint and continues instead of
starting over.


In [ ]:
!pip install -q transformers==4.* torch torchvision scikit-learn pandas pyarrow huggingface_hub scipy
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check Runtime > Change runtime type")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## Mount Google Drive (checkpoints + datasets persist here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/SatQueryAI')
MODELS_DIR = ROOT / 'models'
DATASETS_DIR = ROOT / 'datasets'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
print("Saving everything under:", ROOT)


---
## 1. Ship Classifier (ShipsNet)

4,000 labeled 80x80 ship/no-ship chips. Trains a small CNN — fast even on CPU,
very fast on GPU.


In [ ]:
from huggingface_hub import hf_hub_download
import shutil

ships_dir = DATASETS_DIR / 'ships'
ships_dir.mkdir(exist_ok=True)
ships_parquet = ships_dir / 'train.parquet'
if not ships_parquet.exists():
    p = hf_hub_download(repo_id='jonathan-roberts1/Ships-In-Satellite-Imagery', repo_type='dataset',
                         filename='data/train-00000-of-00001-d1ce9e81c0dc4604.parquet')
    shutil.copy(p, ships_parquet)
print("Ships dataset ready:", ships_parquet)


In [ ]:
import io, numpy as np, pandas as pd
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

SHIP_MODEL_PATH = MODELS_DIR / 'ship_classifier.pt'

class ShipCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 10 * 10, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

class ShipDataset(Dataset):
    def __init__(self, images, labels):
        self.images, self.labels = images, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        img = Image.open(io.BytesIO(self.images[idx]['bytes'])).convert('RGB')
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return torch.from_numpy(arr).permute(2, 0, 1), self.labels[idx]

df = pd.read_parquet(ships_parquet)
labels = (df['label'] == 0).astype(np.int64).to_numpy()
images = df['image'].tolist()
train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.2, random_state=42, stratify=labels)
train_loader = DataLoader(ShipDataset([images[i] for i in train_idx], labels[train_idx]), batch_size=64, shuffle=True)
val_loader = DataLoader(ShipDataset([images[i] for i in val_idx], labels[val_idx]), batch_size=64)

ship_model = ShipCNN().to(DEVICE)
if SHIP_MODEL_PATH.exists():
    ship_model.load_state_dict(torch.load(SHIP_MODEL_PATH, map_location=DEVICE))
    print("Resumed from existing checkpoint")

optimizer = torch.optim.Adam(ship_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(12):
    ship_model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(ship_model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)

    ship_model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (ship_model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    print(f"Epoch {epoch+1}/12  loss={total_loss/len(train_idx):.4f}  val_acc={correct/total:.4f}")
    torch.save(ship_model.state_dict(), SHIP_MODEL_PATH)

print("Ship classifier saved to", SHIP_MODEL_PATH)


### Hard-negative mining (optional but recommended)

Improves real-world precision by teaching the model that breakwaters, piers,
and artificial islands are NOT ships — the single biggest source of false
positives found during local testing.


In [ ]:
import urllib.request

scenes_dir = DATASETS_DIR / 'ships' / 'scenes'
scenes_dir.mkdir(parents=True, exist_ok=True)
scene_urls = {
    'scene_1.png': 'https://raw.githubusercontent.com/rhammell/shipsnet-detector/master/scenes/scene_1.png',
}
for fname, url in scene_urls.items():
    fpath = scenes_dir / fname
    if not fpath.exists():
        urllib.request.urlretrieve(url, fpath)
print("Scenes ready:", list(scenes_dir.iterdir()))


In [ ]:
CHIP = 80
TIGHT_STRIDE = 6
WIDE_STRIDE = 40

# (region x1,y1,x2,y2, stride) — confirmed ship-free regions from local testing
REGIONS = {
    'scene_1.png': [
        (1150, 1600, 1460, 1790, TIGHT_STRIDE),   # breakwater spit
        (880, 630, 1080, 820, TIGHT_STRIDE),      # pier / dock pilings
        (1300, 1500, 1460, 1660, TIGHT_STRIDE),   # pier corner
        (0, 0, 550, 1777, WIDE_STRIDE),           # dense city / land
    ],
}

def extract_patches(image_path, regions):
    image = Image.open(image_path).convert('RGB')
    w, h = image.size
    arr = np.asarray(image, dtype=np.uint8)
    patches = []
    for x1, y1, x2, y2, stride in regions:
        x2, y2 = min(x2, w), min(y2, h)
        for y in range(y1, y2 - CHIP + 1, stride):
            for x in range(x1, x2 - CHIP + 1, stride):
                patches.append(arr[y:y+CHIP, x:x+CHIP])
    return patches

hard_neg_path = DATASETS_DIR / 'ships' / 'hard_negatives.npy'
all_patches = []
for fname, regions in REGIONS.items():
    fpath = scenes_dir / fname
    if fpath.exists():
        all_patches.extend(extract_patches(fpath, regions))
if all_patches:
    np.save(hard_neg_path, np.stack(all_patches))
    print(f"Saved {len(all_patches)} hard negatives")


In [ ]:
# Retrain with hard negatives + class weighting
MAX_HARD_NEG = 2500
hard = np.load(hard_neg_path) if hard_neg_path.exists() else None
if hard is not None:
    if len(hard) > MAX_HARD_NEG:
        idx = np.random.RandomState(42).choice(len(hard), MAX_HARD_NEG, replace=False)
        hard = hard[idx]
    hard_labels = np.zeros(len(hard), dtype=np.int64)

    all_images = images + [None] * len(hard)  # placeholder, handled via combined dataset below

    class CombinedShipDataset(Dataset):
        def __init__(self, orig_images, orig_labels, hard_patches, hard_labels):
            self.orig_images, self.orig_labels = orig_images, orig_labels
            self.hard_patches, self.hard_labels = hard_patches, hard_labels
        def __len__(self):
            return len(self.orig_labels) + len(self.hard_labels)
        def __getitem__(self, idx):
            if idx < len(self.orig_labels):
                img = Image.open(io.BytesIO(self.orig_images[idx]['bytes'])).convert('RGB')
                arr = np.asarray(img, dtype=np.float32) / 255.0
                return torch.from_numpy(arr).permute(2, 0, 1), self.orig_labels[idx]
            j = idx - len(self.orig_labels)
            arr = self.hard_patches[j].astype(np.float32) / 255.0
            return torch.from_numpy(arr).permute(2, 0, 1), self.hard_labels[j]

    full_labels = np.concatenate([labels, hard_labels])
    tr_idx, va_idx = train_test_split(np.arange(len(full_labels)), test_size=0.2, random_state=42, stratify=full_labels)

    ds = CombinedShipDataset(images, labels, hard, hard_labels)
    train_loader2 = DataLoader(torch.utils.data.Subset(ds, tr_idx), batch_size=64, shuffle=True)
    val_loader2 = DataLoader(torch.utils.data.Subset(ds, va_idx), batch_size=64)

    counts = np.bincount(full_labels[tr_idx], minlength=2)
    class_weights = torch.tensor(counts.sum() / (2.0 * counts), dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(ship_model.parameters(), lr=1e-3)

    for epoch in range(12):
        ship_model.train()
        for x, y in train_loader2:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(ship_model(x), y)
            loss.backward()
            optimizer.step()
        ship_model.eval()
        correct = total = tp = fp = fn = 0
        with torch.no_grad():
            for x, y in val_loader2:
                x, y = x.to(DEVICE), y.to(DEVICE)
                pred = ship_model(x).argmax(1)
                correct += (pred == y).sum().item(); total += y.size(0)
                tp += ((pred==1)&(y==1)).sum().item(); fp += ((pred==1)&(y==0)).sum().item(); fn += ((pred==0)&(y==1)).sum().item()
        prec = tp/(tp+fp) if tp+fp else 0; rec = tp/(tp+fn) if tp+fn else 0
        print(f"Epoch {epoch+1}/12  val_acc={correct/total:.4f}  ship_precision={prec:.4f}  ship_recall={rec:.4f}")
        torch.save(ship_model.state_dict(), SHIP_MODEL_PATH)
    print("Hard-negative-mined ship classifier saved.")
else:
    print("No hard negatives found — skipping this step.")


---
## 2. Aerial Multi-Class Detector (NWPU VHR-10)

800 real aerial images, 10 classes: airplane, ship, storage tank, baseball
diamond, tennis court, basketball court, ground track field, harbor, bridge,
vehicle. Fine-tunes a COCO-pretrained Faster R-CNN (MobileNetV3-FPN).


In [ ]:
from huggingface_hub import snapshot_download

nwpu_dir = DATASETS_DIR / 'nwpu_vhr10'
if not (nwpu_dir / 'NWPU VHR-10 dataset').exists():
    snapshot_download(repo_id='SGhazalS/nwpu-vhr10', repo_type='dataset', local_dir=str(nwpu_dir))
print("NWPU VHR-10 ready:", nwpu_dir)


In [ ]:
import re
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NWPU_IMAGES = nwpu_dir / 'NWPU VHR-10 dataset' / 'positive image set'
NWPU_GT = nwpu_dir / 'NWPU VHR-10 dataset' / 'ground truth'
AERIAL_MODEL_PATH = MODELS_DIR / 'aerial_detector.pt'
AERIAL_PROGRESS = MODELS_DIR / 'aerial_detector.progress'
NUM_CLASSES = 11  # 10 objects + background
BOX_RE = re.compile(r'\((\d+),(\d+)\),\((\d+),(\d+)\),(\d+)')

def parse_gt(txt_path):
    boxes, lbls = [], []
    for line in txt_path.read_text().splitlines():
        m = BOX_RE.search(line)
        if not m:
            continue
        x1, y1, x2, y2, cls = map(int, m.groups())
        if x2 <= x1 or y2 <= y1:
            continue
        boxes.append([x1, y1, x2, y2]); lbls.append(cls)
    return boxes, lbls

class NWPUDataset(Dataset):
    def __init__(self, ids):
        self.ids = ids
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        image = Image.open(NWPU_IMAGES / f'{img_id}.jpg').convert('RGB')
        boxes, lbls = parse_gt(NWPU_GT / f'{img_id}.txt')
        tensor = torchvision.transforms.functional.to_tensor(image)
        boxes_t = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.tensor(lbls, dtype=torch.int64).reshape(-1)
        return tensor, {'boxes': boxes_t, 'labels': labels_t, 'image_id': torch.tensor([idx])}

def collate_fn(batch):
    return tuple(zip(*batch))

def build_model():
    m = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights='DEFAULT')
    in_features = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return m

ids = sorted(p.stem for p in NWPU_IMAGES.glob('*.jpg'))
split = int(len(ids) * 0.85)
train_ids, val_ids = ids[:split], ids[split:]
aerial_train_loader = DataLoader(NWPUDataset(train_ids), batch_size=4, shuffle=True, collate_fn=collate_fn)
aerial_val_loader = DataLoader(NWPUDataset(val_ids), batch_size=2, shuffle=False, collate_fn=collate_fn)
print(f"Train: {len(train_ids)}  Val: {len(val_ids)}")


In [ ]:
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
    return inter / union if union > 0 else 0.0

@torch.no_grad()
def evaluate(model, loader, score_thresh=0.5, iou_thresh=0.5):
    model.eval()
    tp = fp = fn = 0
    for imgs, targets in loader:
        imgs = [im.to(DEVICE) for im in imgs]
        preds = model(imgs)
        for pred, target in zip(preds, targets):
            gt_boxes = target['boxes'].tolist(); gt_labels = target['labels'].tolist()
            matched = [False] * len(gt_boxes)
            keep = pred['scores'] >= score_thresh
            pb = pred['boxes'][keep].tolist(); pl = pred['labels'][keep].tolist()
            for pbox, plabel in zip(pb, pl):
                best_iou, best_j = 0.0, -1
                for j, (gbox, glabel) in enumerate(zip(gt_boxes, gt_labels)):
                    if matched[j] or glabel != plabel:
                        continue
                    iou = box_iou(pbox, gbox)
                    if iou > best_iou:
                        best_iou, best_j = iou, j
                if best_iou >= iou_thresh:
                    matched[best_j] = True; tp += 1
                else:
                    fp += 1
            fn += matched.count(False)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return precision, recall

aerial_model = build_model().to(DEVICE)
start_epoch = 0
if AERIAL_MODEL_PATH.exists():
    aerial_model.load_state_dict(torch.load(AERIAL_MODEL_PATH, map_location=DEVICE))
    if AERIAL_PROGRESS.exists():
        start_epoch = int(AERIAL_PROGRESS.read_text().strip())
    print(f"Resumed from epoch {start_epoch}")

params = [p for p in aerial_model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=6, gamma=0.1)
for _ in range(start_epoch):
    lr_scheduler.step()

EPOCHS = 15
for epoch in range(start_epoch, EPOCHS):
    aerial_model.train()
    total_loss, n_batches = 0.0, 0
    for imgs, targets in aerial_train_loader:
        imgs = [im.to(DEVICE) for im in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = aerial_model(imgs, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item(); n_batches += 1
        if n_batches % 30 == 0:
            torch.save(aerial_model.state_dict(), AERIAL_MODEL_PATH)
            AERIAL_PROGRESS.write_text(str(epoch))
            print(f"  mid-epoch checkpoint ({n_batches}/{len(aerial_train_loader)})")

    lr_scheduler.step()
    precision, recall = evaluate(aerial_model, aerial_val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={total_loss/n_batches:.4f}  val_precision={precision:.4f}  val_recall={recall:.4f}")
    torch.save(aerial_model.state_dict(), AERIAL_MODEL_PATH)
    AERIAL_PROGRESS.write_text(str(epoch + 1))

print("Aerial detector training complete.")


---
## 3. Caption Fine-tune (RSICD)

Fixes generic photo-captioning on satellite images (e.g. "a blue background
with a square" instead of a real description) by fine-tuning BLIP on ~11k
real satellite image/caption pairs.


In [ ]:
rsicd_dir = DATASETS_DIR / 'rsicd'
if not (rsicd_dir / 'data' / 'train-00000-of-00001.parquet').exists():
    snapshot_download(repo_id='arampacha/rsicd', repo_type='dataset', local_dir=str(rsicd_dir))
print("RSICD ready:", rsicd_dir)


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

CAPTION_OUT = MODELS_DIR / 'caption_finetuned'
CAPTION_SUBSET = 4000  # GPU can handle far more than the local CPU run could
CAPTION_EPOCHS = 3
CAPTION_BATCH = 8  # bump this up if you have a bigger GPU

df_cap = pd.read_parquet(rsicd_dir / 'data' / 'train-00000-of-00001.parquet')
if len(df_cap) > CAPTION_SUBSET:
    df_cap = df_cap.sample(n=CAPTION_SUBSET, random_state=42).reset_index(drop=True)

class RSICDDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        image = Image.open(io.BytesIO(row['image']['bytes'])).convert('RGB')
        return image, row['captions'][0]

def collate_caption(batch, processor):
    images, captions = zip(*batch)
    inputs = processor(images=list(images), text=list(captions), padding=True, return_tensors='pt')
    inputs['labels'] = inputs['input_ids'].clone()
    return inputs

progress_file = CAPTION_OUT / 'progress.txt'
start_epoch = 0
if progress_file.exists():
    source = str(CAPTION_OUT)
    start_epoch = int(progress_file.read_text().strip())
    print(f"Resuming caption fine-tune from epoch {start_epoch}")
else:
    source = 'Salesforce/blip-image-captioning-base'

cap_processor = BlipProcessor.from_pretrained(source)
cap_model = BlipForConditionalGeneration.from_pretrained(source).to(DEVICE)

cap_loader = DataLoader(RSICDDataset(df_cap), batch_size=CAPTION_BATCH, shuffle=True,
                         collate_fn=lambda b: collate_caption(b, cap_processor))
cap_optimizer = torch.optim.AdamW(cap_model.parameters(), lr=5e-6)

cap_model.train()
for epoch in range(start_epoch, CAPTION_EPOCHS):
    total_loss, n = 0.0, 0
    for batch in cap_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = cap_model(**batch).loss
        cap_optimizer.zero_grad(); loss.backward(); cap_optimizer.step()
        total_loss += loss.item(); n += 1
        if n % 50 == 0:
            print(f"  epoch {epoch+1} batch {n}/{len(cap_loader)}  loss={loss.item():.4f}")
            CAPTION_OUT.mkdir(parents=True, exist_ok=True)
            cap_model.save_pretrained(CAPTION_OUT); cap_processor.save_pretrained(CAPTION_OUT)
            progress_file.write_text(str(epoch))
    print(f"Epoch {epoch+1}/{CAPTION_EPOCHS}  avg_loss={total_loss/n:.4f}")
    CAPTION_OUT.mkdir(parents=True, exist_ok=True)
    cap_model.save_pretrained(CAPTION_OUT); cap_processor.save_pretrained(CAPTION_OUT)
    progress_file.write_text(str(epoch + 1))

print("Caption fine-tune complete:", CAPTION_OUT)


---
## 4. VQA Fine-tune (RSVQA-HR)

Fixes generic VQA answers on satellite images by fine-tuning BLIP-VQA on
real Sentinel-2/aerial imagery Q&A pairs (counting, presence, comparison
questions).


In [ ]:
# Only fetch the shards we'll actually train on (563 total exist upstream;
# downloading all of them would pull way more than this notebook uses).
rsvqa_dir = DATASETS_DIR / 'rsvqa_hr'
VQA_N_SHARDS = 8
shard_patterns = [f'data/train-{i:05d}-of-00563.parquet' for i in range(VQA_N_SHARDS)]
has_shards = (rsvqa_dir / 'data').exists() and len(list((rsvqa_dir / 'data').glob('train-*.parquet'))) >= VQA_N_SHARDS
if not has_shards:
    snapshot_download(repo_id='cpratikaki/RSVQA-HR_qwen_finetuning', repo_type='dataset', local_dir=str(rsvqa_dir),
                       allow_patterns=shard_patterns)
print("RSVQA-HR ready:", rsvqa_dir)


In [ ]:
from transformers import BlipForQuestionAnswering

VQA_OUT = MODELS_DIR / 'vqa_finetuned'
VQA_N_SHARDS = 8       # GPU can handle more shards than the local CPU run could
VQA_SUBSET = 6000
VQA_EPOCHS = 3
VQA_BATCH = 8

shard_files = sorted((rsvqa_dir / 'data').glob('train-*.parquet'))[:VQA_N_SHARDS]
df_vqa = pd.concat([pd.read_parquet(f) for f in shard_files], ignore_index=True)
if len(df_vqa) > VQA_SUBSET:
    df_vqa = df_vqa.sample(n=VQA_SUBSET, random_state=42).reset_index(drop=True)

class RSVQADataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        image = Image.open(io.BytesIO(row['image']['bytes'])).convert('RGB')
        return image, row['question'], str(row['answer'])

def collate_vqa(batch, processor):
    images, questions, answers = zip(*batch)
    inputs = processor(images=list(images), text=list(questions), padding=True, return_tensors='pt')
    inputs['labels'] = processor.tokenizer(list(answers), padding=True, return_tensors='pt').input_ids
    return inputs

vqa_progress_file = VQA_OUT / 'progress.txt'
start_epoch = 0
if vqa_progress_file.exists():
    source = str(VQA_OUT)
    start_epoch = int(vqa_progress_file.read_text().strip())
    print(f"Resuming VQA fine-tune from epoch {start_epoch}")
else:
    source = 'Salesforce/blip-vqa-base'

vqa_processor = BlipProcessor.from_pretrained(source)
vqa_model = BlipForQuestionAnswering.from_pretrained(source).to(DEVICE)

vqa_loader = DataLoader(RSVQADataset(df_vqa), batch_size=VQA_BATCH, shuffle=True,
                         collate_fn=lambda b: collate_vqa(b, vqa_processor))
vqa_optimizer = torch.optim.AdamW(vqa_model.parameters(), lr=5e-6)

vqa_model.train()
for epoch in range(start_epoch, VQA_EPOCHS):
    total_loss, n = 0.0, 0
    for batch in vqa_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = vqa_model(**batch).loss
        vqa_optimizer.zero_grad(); loss.backward(); vqa_optimizer.step()
        total_loss += loss.item(); n += 1
        if n % 50 == 0:
            print(f"  epoch {epoch+1} batch {n}/{len(vqa_loader)}  loss={loss.item():.4f}")
            VQA_OUT.mkdir(parents=True, exist_ok=True)
            vqa_model.save_pretrained(VQA_OUT); vqa_processor.save_pretrained(VQA_OUT)
            vqa_progress_file.write_text(str(epoch))
    print(f"Epoch {epoch+1}/{VQA_EPOCHS}  avg_loss={total_loss/n:.4f}")
    VQA_OUT.mkdir(parents=True, exist_ok=True)
    vqa_model.save_pretrained(VQA_OUT); vqa_processor.save_pretrained(VQA_OUT)
    vqa_progress_file.write_text(str(epoch + 1))

print("VQA fine-tune complete:", VQA_OUT)


---
## 5. Phase 2 — DOTA v1.5 (16-class oriented detection, axis-aligned simplified)

2,806 real aerial images, 16 categories including container crane, roundabout,
swimming pool, helicopter — much broader than NWPU's 10. DOTA's native boxes
are **oriented** (`[cx, cy, w, h, angle]`); this cell converts them to
axis-aligned boxes (min/max of the 4 rotated corners) so it can reuse the
same Faster R-CNN pipeline as the NWPU detector. This trades away rotation
precision for simplicity — a documented tradeoff, not a hidden one. A true
oriented-box detector (e.g. RoI Transformer) is the natural next upgrade.


In [ ]:
dota_dir = DATASETS_DIR / 'dota'
if not (dota_dir / 'data').exists():
    snapshot_download(repo_id='benjamintli/dota-v1.5', repo_type='dataset', local_dir=str(dota_dir),
                       allow_patterns=['data/*.parquet'])
print("DOTA v1.5 ready:", dota_dir)

DOTA_CLASSES = ['background', 'plane', 'ship', 'storage tank', 'baseball diamond', 'tennis court',
                'basketball court', 'ground track field', 'harbor', 'bridge', 'large vehicle',
                'small vehicle', 'helicopter', 'roundabout', 'soccer ball field', 'swimming pool',
                'container crane']
DOTA_NUM_CLASSES = len(DOTA_CLASSES)  # 16 objects + background


In [ ]:
import math

def rotated_to_aabb(cx, cy, w, h, angle_deg):
    """Rotated box [cx,cy,w,h,angle] -> axis-aligned [x1,y1,x2,y2]."""
    angle = math.radians(angle_deg)
    dx, dy = w / 2, h / 2
    corners = [(-dx, -dy), (dx, -dy), (dx, dy), (-dx, dy)]
    cos_a, sin_a = math.cos(angle), math.sin(angle)
    xs, ys = [], []
    for lx, ly in corners:
        rx = lx * cos_a - ly * sin_a + cx
        ry = lx * sin_a + ly * cos_a + cy
        xs.append(rx); ys.append(ry)
    return min(xs), min(ys), max(xs), max(ys)

class DOTADataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        image = Image.open(io.BytesIO(row['image']['bytes'])).convert('RGB')
        w_img, h_img = image.size
        boxes, labels = [], []
        for (cx, cy, bw, bh, ang), cat in zip(row['objects']['bbox_with_angle'], row['objects']['category']):
            x1, y1, x2, y2 = rotated_to_aabb(cx, cy, bw, bh, ang)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w_img, x2), min(h_img, y2)
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            labels.append(int(cat) + 1)  # +1: 0 is background for Faster R-CNN
        tensor = torchvision.transforms.functional.to_tensor(image)
        boxes_t = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.tensor(labels, dtype=torch.int64).reshape(-1)
        return tensor, {'boxes': boxes_t, 'labels': labels_t, 'image_id': torch.tensor([idx])}

DOTA_MODEL_PATH = MODELS_DIR / 'dota_detector.pt'
DOTA_PROGRESS = MODELS_DIR / 'dota_detector.progress'
DOTA_TRAIN_SUBSET = 1500  # cap for reasonable Colab session length

train_files = sorted(dota_dir.glob('data/train-*.parquet'))
df_dota = pd.concat([pd.read_parquet(f) for f in train_files], ignore_index=True)
if len(df_dota) > DOTA_TRAIN_SUBSET:
    df_dota = df_dota.sample(n=DOTA_TRAIN_SUBSET, random_state=42).reset_index(drop=True)
val_files = sorted(dota_dir.glob('data/validation-*.parquet'))
df_dota_val = pd.concat([pd.read_parquet(f) for f in val_files], ignore_index=True).sample(
    n=min(200, sum(len(pd.read_parquet(f)) for f in val_files)), random_state=42).reset_index(drop=True)

dota_train_loader = DataLoader(DOTADataset(df_dota), batch_size=4, shuffle=True, collate_fn=collate_fn)
dota_val_loader = DataLoader(DOTADataset(df_dota_val), batch_size=2, shuffle=False, collate_fn=collate_fn)
print(f"DOTA train: {len(df_dota)}  val: {len(df_dota_val)}")


In [ ]:
def build_dota_model():
    m = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_fpn(weights='DEFAULT')
    in_features = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_features, DOTA_NUM_CLASSES)
    return m

dota_model = build_dota_model().to(DEVICE)
start_epoch = 0
if DOTA_MODEL_PATH.exists():
    dota_model.load_state_dict(torch.load(DOTA_MODEL_PATH, map_location=DEVICE))
    if DOTA_PROGRESS.exists():
        start_epoch = int(DOTA_PROGRESS.read_text().strip())
    print(f"Resumed from epoch {start_epoch}")

dota_params = [p for p in dota_model.parameters() if p.requires_grad]
dota_optimizer = torch.optim.SGD(dota_params, lr=0.005, momentum=0.9, weight_decay=5e-4)
dota_scheduler = torch.optim.lr_scheduler.StepLR(dota_optimizer, step_size=4, gamma=0.1)
for _ in range(start_epoch):
    dota_scheduler.step()

DOTA_EPOCHS = 10
for epoch in range(start_epoch, DOTA_EPOCHS):
    dota_model.train()
    total_loss, n_batches = 0.0, 0
    for imgs, targets in dota_train_loader:
        imgs = [im.to(DEVICE) for im in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = dota_model(imgs, targets)
        loss = sum(loss_dict.values())
        dota_optimizer.zero_grad(); loss.backward(); dota_optimizer.step()
        total_loss += loss.item(); n_batches += 1
        if n_batches % 30 == 0:
            torch.save(dota_model.state_dict(), DOTA_MODEL_PATH)
            DOTA_PROGRESS.write_text(str(epoch))
            print(f"  mid-epoch checkpoint ({n_batches}/{len(dota_train_loader)})")

    dota_scheduler.step()
    precision, recall = evaluate(dota_model, dota_val_loader)
    print(f"Epoch {epoch+1}/{DOTA_EPOCHS}  loss={total_loss/n_batches:.4f}  val_precision={precision:.4f}  val_recall={recall:.4f}")
    torch.save(dota_model.state_dict(), DOTA_MODEL_PATH)
    DOTA_PROGRESS.write_text(str(epoch + 1))

print("DOTA detector training complete.")


---
## 6. Phase 2 — LEVIR-CD (learned change detection)

Replaces/complements the backend's classical image-differencing
change-detection service with a genuinely trained model: a small U-Net takes
a before/after image pair and predicts a pixel-level change mask, trained
on 637 real Google Earth image pairs with human-annotated change masks.


In [ ]:
levircd_dir = DATASETS_DIR / 'levircd'
if not (levircd_dir / 'data').exists():
    snapshot_download(repo_id='ericyu/LEVIRCD_Cropped_256', repo_type='dataset', local_dir=str(levircd_dir))
print("LEVIR-CD ready:", levircd_dir)


In [ ]:
class ChangeUNet(nn.Module):
    """Small U-Net: 6-channel (before+after RGB) in, 1-channel change mask out."""
    def __init__(self):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(cout, cout, 3, padding=1), nn.ReLU())
        self.enc1 = block(6, 32)
        self.enc2 = block(32, 64)
        self.enc3 = block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = block(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(e3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

class LEVIRCDDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        a = Image.open(io.BytesIO(row['imageA']['bytes'])).convert('RGB').resize((256, 256))
        b = Image.open(io.BytesIO(row['imageB']['bytes'])).convert('RGB').resize((256, 256))
        m = Image.open(io.BytesIO(row['label']['bytes'])).convert('L').resize((256, 256))
        a_t = torch.from_numpy(np.asarray(a, dtype=np.float32) / 255.0).permute(2, 0, 1)
        b_t = torch.from_numpy(np.asarray(b, dtype=np.float32) / 255.0).permute(2, 0, 1)
        m_t = torch.from_numpy((np.asarray(m, dtype=np.float32) > 127).astype(np.float32)).unsqueeze(0)
        return torch.cat([a_t, b_t], dim=0), m_t

CD_MODEL_PATH = MODELS_DIR / 'change_detector.pt'
cd_train_files = sorted(levircd_dir.glob('data/train-*.parquet'))
df_cd = pd.concat([pd.read_parquet(f) for f in cd_train_files], ignore_index=True)
CD_SUBSET = 2000
if len(df_cd) > CD_SUBSET:
    df_cd = df_cd.sample(n=CD_SUBSET, random_state=42).reset_index(drop=True)

cd_loader = DataLoader(LEVIRCDDataset(df_cd), batch_size=8, shuffle=True)

cd_model = ChangeUNet().to(DEVICE)
if CD_MODEL_PATH.exists():
    cd_model.load_state_dict(torch.load(CD_MODEL_PATH, map_location=DEVICE))
    print("Resumed from existing checkpoint")

cd_optimizer = torch.optim.Adam(cd_model.parameters(), lr=1e-3)
cd_criterion = nn.BCEWithLogitsLoss()

CD_EPOCHS = 8
for epoch in range(CD_EPOCHS):
    cd_model.train()
    total_loss, n = 0.0, 0
    for x, y in cd_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        cd_optimizer.zero_grad()
        loss = cd_criterion(cd_model(x), y)
        loss.backward(); cd_optimizer.step()
        total_loss += loss.item(); n += 1
        if n % 50 == 0:
            torch.save(cd_model.state_dict(), CD_MODEL_PATH)
            print(f"  epoch {epoch+1} batch {n}/{len(cd_loader)}  loss={loss.item():.4f}  (checkpoint saved)")
    print(f"Epoch {epoch+1}/{CD_EPOCHS}  avg_loss={total_loss/n:.4f}")
    torch.save(cd_model.state_dict(), CD_MODEL_PATH)

print("Change-detection U-Net training complete:", CD_MODEL_PATH)


---
## Not automated in this notebook (manual setup required)

- **DIOR / DIOR-R** — redundant with DOTA's coverage (both are general
  aerial detection benchmarks); skipped to avoid training two similar
  models. Revisit if DOTA's axis-aligned simplification proves insufficient.
- **HRSC2016** — official distribution is Google Drive / Baidu Pan, not a
  script-friendly download. Get it from the official release page and
  adapt the DOTA cell above (same axis-aligned-simplification approach
  works, just swap the annotation parser).
- **xBD** — requires registration at xview2.org before download; cannot be
  automated. Once downloaded, its pre/post-disaster + damage-level format
  is naturally suited to the same U-Net pattern used for LEVIR-CD above,
  extended to multi-class (undamaged/damaged/destroyed) output.
- **SSDD (SAR)** — no verified, reliable download source was found. Worth
  revisiting for the SAR/RISAT roadmap story, but not worth guessing a URL
  that might silently 404 or serve the wrong dataset.


---
## Done — using these checkpoints in the local backend

All checkpoints are saved in your Google Drive under
`MyDrive/SatQueryAI/models/`:

- `ship_classifier.pt`
- `aerial_detector.pt`
- `caption_finetuned/` (folder)
- `vqa_finetuned/` (folder)

**To use them locally:** download that `models/` folder from Drive and copy
its contents into `satquery-ai/backend/../models/` (or `models/` at the
project root, matching the existing layout), replacing the CPU-trained
versions. The backend's `caption_service.py` / `vqa_service.py` load from
`Salesforce/blip-...-base` by default — point them at the local
`caption_finetuned` / `vqa_finetuned` folder paths instead to use the
fine-tuned versions.
